In [1]:
# imports
import pandas as pd
import numpy as np

In [2]:
# USER CONFIGURATION
# Change these paths

# Source data
ORIGINAL_DATA_PATH = "tammireitit_mapping.csv"
# Deduplication data [source_id, found_lipas_id]
DEDUP_PATH = "final_matching_df.csv"
# Mapping table 
MAPPING_PATH = "tammireitit_lipas_type_mapping.csv"

original_df = pd.read_csv(ORIGINAL_DATA_PATH)
dedup_df = pd.read_csv(DEDUP_PATH)
mapping_df = pd.read_csv(MAPPING_PATH, sep = ";")

print("Original rows:", original_df.shape)
print("Dedup rows:", dedup_df.shape)
print("Mapping rows:", mapping_df.shape)

display(original_df.head())
display(dedup_df.head())
display(mapping_df.head())

Original rows: (347, 15)
Dedup rows: (347, 2)
Mapping rows: (25, 6)


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,rotate,source_id,y_eureffin,geometry,x_eureffin
0,Muu,Yksityinen / yhdistys,Pysakointi_liikeste,NaN,Kaarina,Tuorlan esteetön luontopolku,Pysäköintialue,Avaruuspuisto Väisälä,NaN,NaN,NaN,1,6.706329e+06,Point (249254.10773236 6706328.87667998),249254.107732
1,NaN,NaN,Luontotorni,NaN,Lieto,Nautelankosken luonnonsuojelualueen polut / Au...,Näkötorni,Nautelankoski,NaN,NaN,NaN,2,6.721945e+06,Point (251156.5587388 6721945.29509979),251156.558739
2,NaN,NaN,Taukopaikka,NaN,Lieto,Ankka-Nautela luontopolku,Taukopaikka,NaN,NaN,NaN,NaN,3,6.719839e+06,Point (251867.14358549 6719839.44475257),251867.143585
3,NaN,NaN,Portaat,NaN,Lieto,Vanhalinnan Ystävänpolku,Portaat,Vanhalinna,NaN,NaN,NaN,4,6.713795e+06,Point (245642.85277503 6713795.08835029),245642.852775
4,NaN,NaN,Tulipaikka,NaN,Lieto,Ankka-Nautela luontopolku,Laavu,Liesan maja,NaN,NaN,NaN,5,6.719120e+06,Point (252165.38334025 6719120.08476801),252165.383340


,my_id,mapped_lipas_id
0,1,0
1,2,610047
2,3,0
3,4,0
4,5,0


,id,tammireitti_tyyppi,lipas_type_name,lipas_type_code,confirmed,lipas_loi_type
0,1,Pysakointi_liikeste,NaN,NaN,False,parking-spot
1,2,Luontotorni,Luontotorni,204.0,False,NaN
2,3,Taukopaikka,NaN,NaN,False,rest-area
3,4,Portaat,NaN,NaN,False,stairs
4,5,Tulipaikka,NaN,NaN,False,fire-pit


In [3]:
# USER CONFIGURATION
# Add here the column names used in this mapping workflow

# Original source data columns
SOURCE_ID_COL = "source_id"      # Stable unique identifier in the original data
SOURCE_TYPE_COL = "tyyppi"       # Original category/type
SOURCE_NAME_COL = "kohde"        # Original object name
SOURCE_INFO_COL = "lisatieto"    # Additional information from original data 

# Deduplication result columns
DEDUP_SOURCE_ID_COL = "my_id"              # Same identifier as source_id, coming from dedup result
DEDUP_LIPAS_ID_COL = "mapped_lipas_id"     # Existing Lipas id > 0 means duplicate / already exists in LIPAS

# Category mapping table columns
MAP_SOURCE_TYPE_COL = "tammireitti_tyyppi" # Category key in mapping table, matched to original tyyppi
MAP_LIPAS_NAME_COL = "lipas_type_name"     # Lipas category name
MAP_LIPAS_CODE_COL = "lipas_type_code"     # Numeric Lipas category code
MAP_LOI_TYPE_COL = "lipas_loi_type"        # Lipas LOI category
MAP_NEEDS_CHECK_COL = "confirmed"          # True means this category still needs manual handling

In [4]:
# Join deduplication result to original data
working_df = original_df.merge(
    dedup_df,
    how = "left",
    left_on = SOURCE_ID_COL,
    right_on = DEDUP_SOURCE_ID_COL
)

print("Rows after dedup join:", working_df.shape)

display(working_df.head())

Rows after dedup join: (347, 17)


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,rotate,source_id,y_eureffin,geometry,x_eureffin,my_id,mapped_lipas_id
0,Muu,Yksityinen / yhdistys,Pysakointi_liikeste,NaN,Kaarina,Tuorlan esteetön luontopolku,Pysäköintialue,Avaruuspuisto Väisälä,NaN,NaN,NaN,1,6.706329e+06,Point (249254.10773236 6706328.87667998),249254.107732,1,0
1,NaN,NaN,Luontotorni,NaN,Lieto,Nautelankosken luonnonsuojelualueen polut / Au...,Näkötorni,Nautelankoski,NaN,NaN,NaN,2,6.721945e+06,Point (251156.5587388 6721945.29509979),251156.558739,2,610047
2,NaN,NaN,Taukopaikka,NaN,Lieto,Ankka-Nautela luontopolku,Taukopaikka,NaN,NaN,NaN,NaN,3,6.719839e+06,Point (251867.14358549 6719839.44475257),251867.143585,3,0
3,NaN,NaN,Portaat,NaN,Lieto,Vanhalinnan Ystävänpolku,Portaat,Vanhalinna,NaN,NaN,NaN,4,6.713795e+06,Point (245642.85277503 6713795.08835029),245642.852775,4,0
4,NaN,NaN,Tulipaikka,NaN,Lieto,Ankka-Nautela luontopolku,Laavu,Liesan maja,NaN,NaN,NaN,5,6.719120e+06,Point (252165.38334025 6719120.08476801),252165.383340,5,0


In [5]:
# Remove duplicate rows that already exist in LIPAS
duplicates_df = working_df[working_df["mapped_lipas_id"] > 0].copy()
clean_df = working_df[working_df["mapped_lipas_id"] == 0].copy()

print("Duplicate rows removed:", duplicates_df.shape[0])
print("Rows kept for category mapping:", clean_df.shape[0])

display(duplicates_df.head())
display(clean_df.head())

Duplicate rows removed: 26
Rows kept for category mapping: 321


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,rotate,source_id,y_eureffin,geometry,x_eureffin,my_id,mapped_lipas_id
1,NaN,NaN,Luontotorni,NaN,Lieto,Nautelankosken luonnonsuojelualueen polut / Au...,Näkötorni,Nautelankoski,NaN,NaN,NaN,2,6.721945e+06,Point (251156.5587388 6721945.29509979),251156.558739,2,610047
6,Kunta,Kunta / muu,Luontotorni,NaN,Lieto,Littoistenjärven luontopolku / Kuuden kunnanos...,Lintutorni,Järvelä,NaN,NaN,NaN,7,6.711692e+06,Point (246292.61084938 6711691.87185586),246292.610849,7,611083
46,Kunta,Kunta / liikuntatoimi,Vesillelasku,NaN,Lieto,Littoistenjärven melontareitti,Järvelän esteetön melontalaituri,Järvelä,Villa Järvelän palvelut,https://www.jarvela.fi/,NaN,47,6.711389e+06,Point (246478.26718269 6711389.44383875),246478.267183,47,611084
94,Kunta,Kunta / tekninen toimi,Uimaranta,NaN,Paimio,NaN,Ankkalammen maauimala,NaN,NaN,NaN,NaN,95,6.712208e+06,Point (263329.81524975 6712207.85665009),263329.815250,95,100120
108,Kunta,Kunta / tekninen toimi,Uimaranta,Hiekkahelmen uimaranta,Paimio,NaN,Hiekkahelmi,NaN,NaN,NaN,NaN,109,6.705551e+06,Point (263031.86641632 6705551.0612411),263031.866416,109,513259


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,rotate,source_id,y_eureffin,geometry,x_eureffin,my_id,mapped_lipas_id
0,Muu,Yksityinen / yhdistys,Pysakointi_liikeste,NaN,Kaarina,Tuorlan esteetön luontopolku,Pysäköintialue,Avaruuspuisto Väisälä,NaN,NaN,NaN,1,6.706329e+06,Point (249254.10773236 6706328.87667998),249254.107732,1,0
2,NaN,NaN,Taukopaikka,NaN,Lieto,Ankka-Nautela luontopolku,Taukopaikka,NaN,NaN,NaN,NaN,3,6.719839e+06,Point (251867.14358549 6719839.44475257),251867.143585,3,0
3,NaN,NaN,Portaat,NaN,Lieto,Vanhalinnan Ystävänpolku,Portaat,Vanhalinna,NaN,NaN,NaN,4,6.713795e+06,Point (245642.85277503 6713795.08835029),245642.852775,4,0
4,NaN,NaN,Tulipaikka,NaN,Lieto,Ankka-Nautela luontopolku,Laavu,Liesan maja,NaN,NaN,NaN,5,6.719120e+06,Point (252165.38334025 6719120.08476801),252165.383340,5,0
5,NaN,NaN,Tulipaikka,Hamppukota ja tulipaikka,Lieto,Hämeen Härkätie,Kota,Parmaharju,NaN,NaN,NaN,6,6.718069e+06,Point (255896.51124869 6718069.0147056),255896.511249,6,0


In [6]:
# Join category mapping to kept original rows

clean_df = clean_df.merge(
    mapping_df,
    how = "left",
    left_on = SOURCE_TYPE_COL,
    right_on = MAP_SOURCE_TYPE_COL,
    suffixes = ("", "_mapping")
)

print("Rows after category mapping join:", clean_df.shape)
display(clean_df.head())

Rows after category mapping join: (321, 23)


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,...,geometry,x_eureffin,my_id,mapped_lipas_id,id,tammireitti_tyyppi,lipas_type_name,lipas_type_code,confirmed,lipas_loi_type
0,Muu,Yksityinen / yhdistys,Pysakointi_liikeste,NaN,Kaarina,Tuorlan esteetön luontopolku,Pysäköintialue,Avaruuspuisto Väisälä,NaN,NaN,...,Point (249254.10773236 6706328.87667998),249254.107732,1,0,1,Pysakointi_liikeste,NaN,NaN,False,parking-spot
1,NaN,NaN,Taukopaikka,NaN,Lieto,Ankka-Nautela luontopolku,Taukopaikka,NaN,NaN,NaN,...,Point (251867.14358549 6719839.44475257),251867.143585,3,0,3,Taukopaikka,NaN,NaN,False,rest-area
2,NaN,NaN,Portaat,NaN,Lieto,Vanhalinnan Ystävänpolku,Portaat,Vanhalinna,NaN,NaN,...,Point (245642.85277503 6713795.08835029),245642.852775,4,0,4,Portaat,NaN,NaN,False,stairs
3,NaN,NaN,Tulipaikka,NaN,Lieto,Ankka-Nautela luontopolku,Laavu,Liesan maja,NaN,NaN,...,Point (252165.38334025 6719120.08476801),252165.383340,5,0,5,Tulipaikka,NaN,NaN,False,fire-pit
4,NaN,NaN,Tulipaikka,Hamppukota ja tulipaikka,Lieto,Hämeen Härkätie,Kota,Parmaharju,NaN,NaN,...,Point (255896.51124869 6718069.0147056),255896.511249,6,0,5,Tulipaikka,NaN,NaN,False,fire-pit


In [7]:
# Rows marked as needing manual review

needs_category_review_df = clean_df[(clean_df["confirmed"] == True)].copy()
print("Rows needing category review:", needs_category_review_df.shape)

display(needs_category_review_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

Rows needing category review: (18, 23)


,source_id,tyyppi,kohde,lipas_type_name,lipas_type_code,lipas_loi_type,confirmed
14,17,Kahvila,Kartanon päärakennus,NaN,NaN,NaN,True
15,18,Kahvila,Nautelankosken museo,NaN,NaN,NaN,True
43,46,Kahvila,Vilkkimäen meijeri,NaN,NaN,NaN,True
57,61,Kahvila,Ravintola Toivo,NaN,NaN,NaN,True
95,100,Kahvila,Cafe Asta,NaN,NaN,NaN,True
96,101,Kahvila,R-kioski,NaN,NaN,NaN,True
97,102,Kahvila,NaN,NaN,NaN,NaN,True
107,113,Kahvila,Neste Paimio,NaN,NaN,NaN,True
108,114,Kahvila,Iivari cafe & bistro,NaN,NaN,NaN,True
109,115,Kahvila,Heimon grilli,NaN,NaN,NaN,True


In [8]:
# EDIT IF NEEDED
# Add one or many unwanted original data categories to remove 

types_to_remove = ["Kahvila"]

removed_category_rows_df = clean_df[clean_df[SOURCE_TYPE_COL].isin(types_to_remove)].copy()
clean_df = clean_df[~clean_df[SOURCE_TYPE_COL].isin(types_to_remove)].copy()

print("Removed rows:", removed_category_rows_df.shape)
print("Remaining rows:", clean_df.shape)

display(removed_category_rows_df)
display(clean_df.loc[clean_df[MAP_NEEDS_CHECK_COL] == True])

Removed rows: (17, 23)
Remaining rows: (304, 23)


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,...,geometry,x_eureffin,my_id,mapped_lipas_id,id,tammireitti_tyyppi,lipas_type_name,lipas_type_code,confirmed,lipas_loi_type
14,NaN,NaN,Kahvila,Vanhalinna,Lieto,Hämeen Härkätie / Kuuden kunnanosan kierros / ...,Kartanon päärakennus,Liedon Vanhalinna,Vanhalinnan palvelut,https://vanhalinna.fi/,...,Point (246196.54214042 6714480.4595376),246196.542140,17,0,8,Kahvila,NaN,NaN,True,NaN
15,NaN,NaN,Kahvila,Nautelankosken museo,Lieto,Kuuden kunnanosan kierros / Nautelankosken luo...,Nautelankosken museo,Nautelankoski,Nautelankosken museon palvelut,https://www.liedonmuseo.fi/,...,Point (250882.41123647 6722378.42113587),250882.411236,18,0,8,Kahvila,NaN,NaN,True,NaN
43,NaN,NaN,Kahvila,Kahvila Namia,Lieto,Kuuden kunnanosan kierros,Vilkkimäen meijeri,Vilkkimäki,NaN,NaN,...,Point (249078.37354895 6719055.85138483),249078.373549,46,0,8,Kahvila,NaN,NaN,True,NaN
57,Muu,Yksityinen / säätiö,Kahvila,Katso lisätietoja Parantolan ja ravintolan ver...,Paimio,NaN,Ravintola Toivo,NaN,NaN,https://paimiosanatorium.com/fi/syo-nuku/ravin...,...,Point (265576.41162817 6710785.28629848),265576.411628,61,0,8,Kahvila,NaN,NaN,True,NaN
95,Muu,Muu,Kahvila,Katso aukioloajat leipomo-kahvilan omilta verk...,Paimio,NaN,Cafe Asta,NaN,NaN,https://www.cafeasta.fi/,...,Point (262897.28287398 6710119.44340858),262897.282874,100,0,8,Kahvila,NaN,NaN,True,NaN
96,Muu,Muu,Kahvila,Katso aukioloajat kioskin omilta verkkosivuilta,Paimio,NaN,R-kioski,NaN,NaN,https://www.r-kioski.fi/kioskit/paimio-asemati...,...,Point (262884.75975171 6710114.56249524),262884.759752,101,0,8,Kahvila,NaN,NaN,True,NaN
97,Muu,Muu,Kahvila,Katso aukioloajat kahvilan omilta verkkosivuilta,Paimio,NaN,NaN,NaN,NaN,https://www.cafeatsalea.fi/,...,Point (264086.14062923 6715078.06867481),264086.140629,102,0,8,Kahvila,NaN,NaN,True,NaN
107,Muu,Muu,Kahvila,Katso aukioloajat palveluntarjoajan verkkosivu...,Paimio,NaN,Neste Paimio,NaN,NaN,https://www.neste.fi/asema/neste-k-paimio,...,Point (264679.33253681 6706504.65701234),264679.332537,113,0,8,Kahvila,NaN,NaN,True,NaN
108,Muu,Muu,Kahvila,Katso aukioloajat Iivarin verkkosivuilta,Paimio,NaN,Iivari cafe & bistro,NaN,NaN,https://www.iivaricafe.fi/,...,Point (263174.32036418 6709396.60682297),263174.320364,114,0,8,Kahvila,NaN,NaN,True,NaN
109,Muu,Muu,Kahvila,Katso lisätietoja Heimon grillin verkkosivuilta,Paimio,NaN,Heimon grilli,NaN,NaN,https://www.facebook.com/heimongrilli/?locale=...,...,Point (262777.77162609 6710140.28063731),262777.771626,115,0,8,Kahvila,NaN,NaN,True,NaN


,omistaja,yllapitaja,tyyppi,lisatieto,kunta,reitti,kohde,sijainti,palvelut,linkki,...,geometry,x_eureffin,my_id,mapped_lipas_id,id,tammireitti_tyyppi,lipas_type_name,lipas_type_code,confirmed,lipas_loi_type
135,Kunta,Kunta / muu,Aktiviteetti,NaN,Kaarina,NaN,Liikenne- ja leikkipuisto,Hovirinnan rantapuisto,NaN,NaN,...,Point (245313.18251202 6704750.97661256),245313.182512,143,0,18,Aktiviteetti,NaN,NaN,True,NaN


In [9]:
# EDIT IF NEEDED
# Lipas category manual mapping example

# Select one source data category to inspect before manual Lipas category mapping

source_category_to_map = "Aktiviteetti"

selected_category_df = clean_df[clean_df[SOURCE_TYPE_COL] == source_category_to_map].copy()

print("Selected source category:", source_category_to_map)
print("Rows in selected category:", selected_category_df.shape)

with pd.option_context("display.max_rows", None):
    display(selected_category_df.loc[:, [
        SOURCE_ID_COL,
        SOURCE_TYPE_COL,
        SOURCE_NAME_COL,
        SOURCE_INFO_COL,
        MAP_LIPAS_NAME_COL,
        MAP_LIPAS_CODE_COL,
        MAP_LOI_TYPE_COL,
        MAP_NEEDS_CHECK_COL,
    ]])

Selected source category: Aktiviteetti
Rows in selected category: (1, 23)


,source_id,tyyppi,kohde,lisatieto,lipas_type_name,lipas_type_code,lipas_loi_type,confirmed
135,143,Aktiviteetti,Liikenne- ja leikkipuisto,NaN,NaN,NaN,NaN,True


In [10]:
# EDIT IF NEEDED
# Manual mapping for one source category to one or many Lipas categories based on source_id

# One source category can be split into many different LIPAS categories by listing the source_id values that belong to each target Lipas category.
source_category_to_map = "Aktiviteetti"

# Supports [1,n] LIPAS categories
lipas_category_rules = [
    {
        "source_ids": [143],
        "lipas_type_name": "Liikuntapuisto",
        "lipas_type_code": 1110,
    },
    #{
    #  "source_ids": [0],
    #  "lipas_type_name": "Name",
    #  "lipas_type_code": 0000,
    #},
]

for rule in lipas_category_rules:
    mask = ((clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(rule["source_ids"])))

    clean_df.loc[mask, MAP_LIPAS_NAME_COL] = rule["lipas_type_name"]
    clean_df.loc[mask, MAP_LIPAS_CODE_COL] = rule["lipas_type_code"]

    # These rows have now been manually handled
    clean_df.loc[mask, MAP_NEEDS_CHECK_COL] = False

mapped_source_ids = [source_id for rule in lipas_category_rules for source_id in rule["source_ids"]]
mapped_rows_df = clean_df[(clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(mapped_source_ids))].copy()

print("Source category:", source_category_to_map)
print("Mapped rows:", mapped_rows_df.shape)

display(mapped_rows_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

Source category: Aktiviteetti
Mapped rows: (1, 23)


,source_id,tyyppi,kohde,lipas_type_name,lipas_type_code,lipas_loi_type,confirmed
135,143,Aktiviteetti,Liikenne- ja leikkipuisto,Liikuntapuisto,1110.0,NaN,False


In [11]:
# EDIT IF NEEDED
# Lipas LOI manual mapping example

# Select one source data category to inspect before manual Lipas category mapping

# Since most of these were buildings, manual check was set to false in the mapping table and the outliers still checked manually
# This behaviour is supported in the logic, but needs attention as this circumvents the sanity check if done.  
source_category_to_map = "Nahtavyys"

selected_category_df = clean_df[clean_df[SOURCE_TYPE_COL] == source_category_to_map].copy()

print("Selected source category:", source_category_to_map)
print("Rows in selected category:", selected_category_df.shape)

with pd.option_context("display.max_rows", None):
    display(selected_category_df.loc[:, [
        SOURCE_ID_COL,
        SOURCE_TYPE_COL,
        SOURCE_NAME_COL,
        SOURCE_INFO_COL,
        MAP_LIPAS_NAME_COL,
        MAP_LIPAS_CODE_COL,
        MAP_LOI_TYPE_COL,
        MAP_NEEDS_CHECK_COL,
    ]])

Selected source category: Nahtavyys
Rows in selected category: (63, 23)


,source_id,tyyppi,kohde,lisatieto,lipas_type_name,lipas_type_code,lipas_loi_type,confirmed
26,29,Nahtavyys,Littoisten verkatehdas,NaN,NaN,NaN,building,False
27,30,Nahtavyys,Kuoviluoto,NaN,NaN,NaN,building,False
28,31,Nahtavyys,Saukonojan vanha koulu,NaN,NaN,NaN,building,False
29,32,Nahtavyys,Keppolan aitta,NaN,NaN,NaN,building,False
35,38,Nahtavyys,Parmaharju,NaN,NaN,NaN,building,False
36,39,Nahtavyys,Viinamäen hautausmaa,NaN,NaN,NaN,building,False
37,40,Nahtavyys,Pyhän Pietarin kirkko,NaN,NaN,NaN,building,False
40,43,Nahtavyys,Henrik Antinpojan hauta,NaN,NaN,NaN,building,False
42,45,Nahtavyys,Viikinkivene Helka,NaN,NaN,NaN,building,False
54,58,Nahtavyys,NaN,Parantolan metsän kunniataulu,NaN,NaN,building,False


In [12]:
# EDIT IF NEEDED
# Manual mapping for one source category to one or many Lipas LOI categories based on source_id

# One source_category can be split into many different LOI categories by listing the source_id values that belong to each target LOI category.
source_category_to_map = "Nahtavyys"

# Supports [1,n] LOI categories
loi_category_rules = [
    {
        "source_ids": [320, 321, 211, 208, 202, 200, 199, 198, 193, 192, 128, 123, 70, 58, 213],
        "lipas_loi_type": "historical-structure",
    },
    {
        "source_ids": [347, 205, 107, 105, 87, 69, 59, 188],
        "lipas_loi_type": "natural-attraction",
    },
]

for rule in loi_category_rules:
    mask = ((clean_df[SOURCE_TYPE_COL] == source_category_to_map) & (clean_df[SOURCE_ID_COL].isin(rule["source_ids"])))
    clean_df.loc[mask, MAP_LOI_TYPE_COL] = rule["lipas_loi_type"]
    
    # These rows have now been manually handled
    clean_df.loc[mask, MAP_NEEDS_CHECK_COL] = False

mapped_source_ids = [source_id for rule in loi_category_rules for source_id in rule["source_ids"]]
mapped_rows_df = clean_df[(clean_df["tyyppi"] == source_category_to_map) & (clean_df["source_id"].isin(mapped_source_ids))].copy()

print("Source category:", source_category_to_map)
print("Mapped rows:", mapped_rows_df.shape)

display(mapped_rows_df.loc[:, [
    SOURCE_ID_COL,
    SOURCE_TYPE_COL,
    SOURCE_NAME_COL,
    MAP_LIPAS_NAME_COL,
    MAP_LIPAS_CODE_COL,
    MAP_LOI_TYPE_COL,
    MAP_NEEDS_CHECK_COL,
]])

Source category: Nahtavyys
Mapped rows: (23, 23)


,source_id,tyyppi,kohde,lipas_type_name,lipas_type_code,lipas_loi_type,confirmed
54,58,Nahtavyys,NaN,NaN,NaN,historical-structure,False
55,59,Nahtavyys,Lemmenlampi,NaN,NaN,natural-attraction,False
65,69,Nahtavyys,Muinaisranta,NaN,NaN,natural-attraction,False
66,70,Nahtavyys,Korpelan korsu,NaN,NaN,historical-structure,False
83,87,Nahtavyys,Kohiseva,NaN,NaN,natural-attraction,False
100,105,Nahtavyys,Paimion Jokipuisto,NaN,NaN,natural-attraction,False
102,107,Nahtavyys,Parantolan metsä,NaN,NaN,natural-attraction,False
117,123,Nahtavyys,Rekottilan linnavuori,NaN,NaN,historical-structure,False
121,128,Nahtavyys,Spurilan muinaisjäännösalue,NaN,NaN,historical-structure,False
175,188,Nahtavyys,Nautelankoski,NaN,NaN,natural-attraction,False


In [13]:
# Sanity checks before export

print("Original rows:", original_df.shape[0])
print("Duplicate rows removed:", duplicates_df.shape[0])
print("Unwanted rows removed:", removed_category_rows_df.shape[0])
print("\nFinal rows to export:", clean_df.shape[0])

expected_final_rows = original_df.shape[0] - duplicates_df.shape[0] - removed_category_rows_df.shape[0]

print("Expected final rows:", expected_final_rows)
print("Final row count matches expected:", clean_df.shape[0] == expected_final_rows)

print("\nUnique source_id values:", clean_df["source_id"].nunique())
print("Duplicate source_id values:", clean_df["source_id"].duplicated().sum())

print("\nRows still marked as waiting for manual check = True:", (clean_df["confirmed"] == True).sum())

Original rows: 347
Duplicate rows removed: 26
Unwanted rows removed: 17

Final rows to export: 304
Expected final rows: 304
Final row count matches expected: True

Unique source_id values: 304
Duplicate source_id values: 0

Rows still marked as waiting for manual check = True: 0


In [14]:
# CONFIGURE EXPORT
# Export final cleaned and categorized Tammireitit data to CSV

OUTPUT_PATH = "tammireitit_final_cleaned_mapped.csv"

final_export_df = clean_df.copy()

# Append route information from "reitti" to "lisatieto" since this field does not exist in LIPAS

has_reitti = final_export_df["reitti"].fillna("").astype(str).str.strip() != ""

final_export_df.loc[has_reitti, "lisatieto"] = (
    final_export_df.loc[has_reitti, "lisatieto"].fillna("").astype(str).str.strip()
    + " Reitti: "
    + final_export_df.loc[has_reitti, "reitti"].astype(str).str.strip()
).str.strip()


# Remove helper columns from joins if needed
columns_to_drop = [
    "my_id",
    "id",
    "tammireitti_tyyppi",
    "confirmed",
    "rotate",
    "mapped_lipas_id",
    "reitti"
]

final_export_df = final_export_df.drop(columns=[col for col in columns_to_drop if col in final_export_df.columns])

# Required final column order. Optional but helps with readability
preferred_column_order = [
    "source_id",
    "kohde",
    "tyyppi",
    "kunta",
    "lisatieto",
    "palvelut",
    "linkki",
    "lipas_type_name",
    "lipas_type_code",
    "lipas_loi_type",
    "omistaja",
    "yllapitaja",
    "sijainti",
    "x_eureffin",
    "y_eureffin",
    "geometry",
]

# Keep only columns that exist, in the requested order
final_export_df = final_export_df[[col for col in preferred_column_order if col in final_export_df.columns]]

final_export_df.to_csv(OUTPUT_PATH, index=False)

print("Exported final CSV:", OUTPUT_PATH)
print("Rows:", final_export_df.shape[0])
print("Columns:", final_export_df.shape[1])

display(final_export_df.head(10))

Exported final CSV: tammireitit_final_cleaned_mapped.csv
Rows: 304
Columns: 16


,source_id,kohde,tyyppi,kunta,lisatieto,palvelut,linkki,lipas_type_name,lipas_type_code,lipas_loi_type,omistaja,yllapitaja,sijainti,x_eureffin,y_eureffin,geometry
0,1,Pysäköintialue,Pysakointi_liikeste,Kaarina,Reitti: Tuorlan esteetön luontopolku,NaN,NaN,NaN,NaN,parking-spot,Muu,Yksityinen / yhdistys,Avaruuspuisto Väisälä,249254.107732,6.706329e+06,Point (249254.10773236 6706328.87667998)
1,3,Taukopaikka,Taukopaikka,Lieto,Reitti: Ankka-Nautela luontopolku,NaN,NaN,NaN,NaN,rest-area,NaN,NaN,NaN,251867.143585,6.719839e+06,Point (251867.14358549 6719839.44475257)
2,4,Portaat,Portaat,Lieto,Reitti: Vanhalinnan Ystävänpolku,NaN,NaN,NaN,NaN,stairs,NaN,NaN,Vanhalinna,245642.852775,6.713795e+06,Point (245642.85277503 6713795.08835029)
3,5,Laavu,Tulipaikka,Lieto,Reitti: Ankka-Nautela luontopolku,NaN,NaN,NaN,NaN,fire-pit,NaN,NaN,Liesan maja,252165.383340,6.719120e+06,Point (252165.38334025 6719120.08476801)
4,6,Kota,Tulipaikka,Lieto,Hamppukota ja tulipaikka Reitti: Hämeen Härkätie,NaN,NaN,NaN,NaN,fire-pit,NaN,NaN,Parmaharju,255896.511249,6.718069e+06,Point (255896.51124869 6718069.0147056)
5,8,Taukopaikka,Taukopaikka,Lieto,Katettu eväspöytä ja penkit Reitti: Ankka-Naut...,NaN,NaN,NaN,NaN,rest-area,NaN,NaN,Poso,252719.153938,6.721242e+06,Point (252719.15393824 6721241.91135619)
6,9,Portaat,Portaat,Lieto,"Pitkät, puiset portaat Reitti: Nautelankosken ...",NaN,NaN,NaN,NaN,stairs,NaN,NaN,Nautelankoski,251154.856810,6.721922e+06,Point (251154.85681026 6721921.53129717)
7,10,Laituri,Vesillelasku,Lieto,Reitti: Aurajoen melontareitti,NaN,NaN,NaN,NaN,boat-ramp,NaN,NaN,Liedon Vanhalinna,246118.685479,6.714696e+06,Point (246118.68547878 6714696.28847793)
8,11,Pysäköintialue,Pysakointialue,Lieto,Reitti: Ankka-Nautela luontopolku,NaN,NaN,NaN,NaN,parking-spot,NaN,NaN,Ankka,252819.286340,6.718783e+06,Point (252819.28634011 6718782.69654153)
9,12,Pysäköintialue,Pysakointialue,Lieto,Reitti: Ankka-Nautela luontopolku,NaN,NaN,NaN,NaN,parking-spot,NaN,NaN,Pieneläinhautausmaa,251972.218030,6.719507e+06,Point (251972.21802981 6719507.24923929)
